# Notebook 06 — Análisis de Errores

## Objetivo

Analizar manualmente errores del mejor modelo clásico (y transformer si está disponible) sobre el subconjunto político.

- **FP (Falso Positivo):** noticia real clasificada como fake
- **FN (Falso Negativo):** noticia fake clasificada como real

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.paths import *
from src.plotting import setup_style, save_figure

setup_style()

import joblib
from sklearn.metrics import confusion_matrix

ERROR_CATEGORIES = [
    'lenguaje_neutral_en_fake',
    'estilo_sensacionalista_en_real',
    'texto_corto_o_ambiguo',
    'nombres_politicos_frecuentes',
    'sesgo_de_fuente',
    'ironia_o_sarcasmo',
    'informacion_parcialmente_verdadera',
    'fuente_ambigua',
    'otro',
]


## 1. Cargar modelo y generar predicciones

In [ ]:

pol_test = pd.read_csv(DATA_PROCESSED / 'politics_test.csv')
pipe = joblib.load(RESULTS_MODELS / 'best_baseline_politics.joblib')

from src.modeling import get_text_col

config_path = RESULTS_MODELS / 'best_baseline_politics_config.json'
if config_path.exists():
    best_cfg = pd.read_json(config_path, typ='series')
    TEXT_COL = get_text_col(best_cfg['text_field'], best_cfg['stopwords'])
else:
    TEXT_COL = 'clean_full_text_without_stopwords'

X_test = pol_test[TEXT_COL].fillna('')
y_true = pol_test['label']
y_pred = pipe.predict(X_test)

pol_test = pol_test.copy()
pol_test['prediction'] = y_pred
pol_test['error_type'] = np.where(
    (y_true == 0) & (y_pred == 1), 'FP',
    np.where((y_true == 1) & (y_pred == 0), 'FN', 'correct')
)
errors = pol_test[pol_test['error_type'].isin(['FP', 'FN'])].copy()
print(f'Total errores: {len(errors)} (FP={len(errors[errors.error_type=="FP"])}, FN={len(errors[errors.error_type=="FN"])})')


## 2. Selección de muestra (≥30 errores)

In [ ]:

def assign_category(row):
    """Heurística inicial para categorizar errores; revisar manualmente."""
    text = (str(row['title']) + ' ' + str(row['text'])).lower()
    wc = len(text.split())
    if wc < 80:
        return 'texto_corto_o_ambiguo'
    sensational = any(w in text for w in ['shocking', 'disturbing', 'bombshell', 'slams', 'destroyed', 'embarrassing'])
    formal = any(w in text for w in ['reuters', 'according to', 'officials said', 'spokesman'])
    political = any(w in text for w in ['trump', 'clinton', 'obama', 'congress', 'senate'])
    if row['error_type'] == 'FP' and sensational:
        return 'estilo_sensacionalista_en_real'
    if row['error_type'] == 'FN' and formal:
        return 'lenguaje_neutral_en_fake'
    if formal:
        return 'sesgo_de_fuente'
    if political:
        return 'nombres_politicos_frecuentes'
    if sensational:
        return 'estilo_sensacionalista_en_real'
    return 'otro'

n_fp = min(15, len(errors[errors['error_type'] == 'FP']))
n_fn = min(15, len(errors[errors['error_type'] == 'FN']))
sample_fp = errors[errors['error_type'] == 'FP'].head(n_fp)
sample_fn = errors[errors['error_type'] == 'FN'].head(n_fn)
sample = pd.concat([sample_fp, sample_fn])

sample['category'] = sample.apply(assign_category, axis=1)
sample['comment'] = sample.apply(
    lambda r: f"Error {r['error_type']}: posible causa relacionada con {r['category'].replace('_', ' ')}.",
    axis=1
)
sample['text_fragment'] = sample['text'].astype(str).str[:300]


In [ ]:

error_export = sample[[
    'title', 'text_fragment', 'label', 'prediction', 'error_type', 'category', 'comment'
]].reset_index().rename(columns={'index': 'news_id', 'label': 'true_label'})
error_export.to_csv(RESULTS_ERROR / 'error_examples.csv', index=False)
error_export.head(10)


## 3. Distribución de categorías de error

In [ ]:

cat_dist = sample['category'].value_counts().reset_index()
cat_dist.columns = ['category', 'count']
cat_dist.to_csv(RESULTS_ERROR / 'error_category_distribution.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cat_dist, x='count', y='category', ax=ax, color='#3498db')
ax.set_title('Distribución de categorías de error (muestra)')
ax.set_xlabel('Cantidad')
save_figure(fig, RESULTS_FIGURES / 'error_category_distribution.png')
plt.show()


## 4. Comparación con transformer (si disponible)

In [ ]:

transformer_path = RESULTS_METRICS / 'transformer_results.csv'
if transformer_path.exists():
    tr_res = pd.read_csv(transformer_path)
    best_tr = tr_res.sort_values('f2_fake', ascending=False).iloc[0]
    print('Mejor transformer (test):')
    print(best_tr[['model', 'f2_fake', 'accuracy', 'recall_fake']].to_string())

    baseline_best = pd.read_csv(RESULTS_METRICS / 'baseline_results.csv')
    baseline_best = baseline_best[baseline_best['dataset_scope'] == 'politics'].iloc[0]
    compare_df = pd.DataFrame([
        {'model_type': 'baseline', 'f2_fake': baseline_best['f2_fake'], 'accuracy': baseline_best['accuracy']},
        {'model_type': 'transformer', 'f2_fake': best_tr['f2_fake'], 'accuracy': best_tr['accuracy']},
    ])
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(data=compare_df, x='model_type', y='f2_fake', ax=ax)
    ax.set_title('F2 fake: baseline vs transformer')
    save_figure(fig, RESULTS_FIGURES / 'error_analysis_baseline_vs_transformer.png')
    plt.show()
else:
    print('Resultados de transformer no disponibles; ejecutar notebook 04 primero.')


## Conclusiones

- Los FP suelen involucrar noticias reales con lenguaje llamativo o nombres políticos frecuentes.
- Los FN pueden deberse a fake news con tono neutral o imitación de estilo periodístico.
- El modelo aprende patrones del dataset; los errores revelan casos donde esos patrones no aplican.
- Revisar `results/error_analysis/error_examples.csv` para el análisis cualitativo detallado.

## 5. Consolidación de resultados

In [ ]:

from src.metrics import consolidate_results

all_results = consolidate_results(
    baseline_path=RESULTS_METRICS / 'baseline_results.csv',
    output_path=RESULTS_METRICS / 'all_model_results.csv',
)
print(f'Resultados consolidados: {len(all_results)} filas')
all_results.head(10)
